In [1]:
import logging
import shutil
import socket
import traceback
from datetime import UTC, datetime
from math import prod
from pathlib import Path
from pprint import pprint
from time import perf_counter

import numpy as np
from memory_tempfile import MemoryTempfile as MemoryTemper
from tifffile import TiffFile

from comptest import imagemagick, libtiff, run_process
from comptest.constants import DATASETS_DIR, PROJECT_SCRATCH_DIR
from comptest.fs import get_dir_size_bytes
from comptest.types import CompressionTest, FileStackCompressionConfig

%load_ext autoreload
%autoreload 2

In [2]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

In [3]:
PROJECT_DIR = Path().resolve()
OUTPUT_DIR = PROJECT_DIR / "output"
assert OUTPUT_DIR.is_dir()

mem_temper = MemoryTemper(filesystem_types=["tmpfs"])

In [4]:
COMPRESSION_TESTS_DIR = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests"
)
assert COMPRESSION_TESTS_DIR.is_dir()

In [5]:
input_file = COMPRESSION_TESTS_DIR / "uncompressed.tif"
assert input_file.is_file()

output_file = COMPRESSION_TESTS_DIR / "compressed.tif"

In [6]:
run_process("tiffcp", ["-c", "lzma:2:p6", str(input_file), str(output_file)])

ProcessResult(time=ProcessTiming(wall=3.248888705857098, user=3.183807, system=0.031998), memory=ProcessMemoryUsage(max_rss=131055616), returncode=0, stdout='', stderr='')

In [7]:
DATASET = "Canislupus_CZM_712"
DATASET_DIR = DATASETS_DIR / DATASET
assert DATASET_DIR.is_dir()

SCRATCH_INPUT_FILES = list(DATASET_DIR.glob("*.tif"))

In [8]:
# Load input files into memory
# !!! Note: We currently don't clean this up after use!
temp_dir_path = (
    Path(mem_temper.gettempdir()) / "s250250-compression-tests-filestack-in-memory"
)

if temp_dir_path.exists():
    shutil.rmtree(temp_dir_path)

temp_dir_path.mkdir(parents=True, exist_ok=True)
INPUT_FILES = []
for file in SCRATCH_INPUT_FILES:
    shutil.copy(file, temp_dir_path / file.name)
    INPUT_FILES.append(temp_dir_path / file.name)


In [9]:
# Check dtype is the same for all
INPUT_DTYPE = None
for file in INPUT_FILES:
    with TiffFile(file) as tif:
        dtype = tif.pages[0].dtype
        if INPUT_DTYPE is None:
            INPUT_DTYPE = dtype
        else:
            assert dtype == INPUT_DTYPE, (
                f"Input files have different dtypes: {dtype} != {INPUT_DTYPE}"
            )
INPUT_DTYPE

dtype('uint16')

In [10]:
DTYPE_UINT = {
    np.bool,
    np.uint8,
    np.uint16,
    np.uint32,
    np.uint64,
}
DTYPE_SINT = {
    np.int8,
    np.int16,
    np.int32,
    np.int64,
}
DTYPE_INT = DTYPE_UINT.union(DTYPE_SINT)
DTYPE_FLOAT = {
    np.float16,
    np.float32,
    np.float64,
    np.float128,
}
DTYPE_COMPLEX = {
    np.complex64,
    np.complex128,
    np.complex256,
}
DTYPE_ANY = DTYPE_INT.union(DTYPE_FLOAT).union(DTYPE_COMPLEX)

In [11]:
# TESTS DEFINITIONS

# Uncomment below to see options for tiffcp
# !tiffcp --help
TESTS = {
    # ============================== N-BIT LOSSY (LERC z>0) ==============================
    "lerc_z0.5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0.5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z1_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:1:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z2_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:2:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z10_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:10:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z20_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:20:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    # ============================== 8-BIT LOSSLESS ==============================
    # libtiff: None -- tiffcp requires 8-bit source data already; can't downsample inline
    "webp_lossless_pred1_p50": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "50",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p75": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "75",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p100": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "100",
        ],
        "notes": ["lossless", "8bit"],
    },
    "g3_1d": {
        "libtiff": None,  # G3 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Fax"],
        "notes": ["lossless", "1bit"],
    },
    "g3_2d": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # 1D/2D not separately exposed
        "notes": ["lossless", "1bit"],
    },
    "g3_2d_fill": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # fill/EOL byte-align not exposed
        "notes": ["lossless", "1bit"],
    },
    "g4": {
        "libtiff": None,  # G4 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Group4"],
        "notes": ["lossless", "1bit"],
    },
    # ============================== 8-BIT LOSSY ==============================
    "webp_pred2_p25": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p90": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "90"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p100": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "100"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q10": {
        "libtiff": None,  # JPEG codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "10"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q25": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "85"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "95"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q96": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "96"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q97": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "97"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q98": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "98"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q99": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "99"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q100": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "100"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "50",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "75",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "85",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "95",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
}

In [12]:
# TODO:
# - Lossy
# - 8 bit codecs (need dataset if doing this)
# - bilevel codecs (need dataset if doing this)


def _lt_comp(*args):
    return (libtiff.compress_bulk, args, {})


def _im_comp(*args):
    return (imagemagick.compress_bulk, args, {})


# BASE CONFIGS
_BC = {
    "paradigm": "file_stack",
    "lossless": True,
}
_BC_LIBTIFF = _BC | {
    "implementation": "libtiff/tiffcp",
    "decompressor": (libtiff.decompress_bulk, (), {}),
    "comparer": (libtiff.compare_bulk, ("-t",), {}),
}
_BC_IMAGEMAGICK = _BC | {
    "implementation": "imagemagick",
    "decompressor": (imagemagick.decompress_bulk, (), {}),
    "comparer": (imagemagick.compare_bulk, (), {}),
}
_C = FileStackCompressionConfig


BASELINE_CONFIGS = [
    _C(
        name="libtiff_baseline",
        series="tiff-baseline",
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", "none"),
        **_BC_LIBTIFF,
    ),
    _C(
        name="imagemagick_baseline",
        series="tiff-baseline",
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp("-compress", "None"),
        **_BC_IMAGEMAGICK,
    ),
]

# --- LZW ---
_PREDICTOR_NAME_MAP = {1: "none", 2: "horizontal", 3: "float"}
LIBTIFF_LZW_CONFIGS = [
    _C(
        name=f"libtiff_lzw_pred{i}",
        series="tiff-lzw",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"lzw:{i}"),
        **_BC_LIBTIFF,
    )
    for i in [1, 2, 3]  # 1: no differencing
]
IMAGEMAGICK_LZW_CONFIGS = [
    _C(
        name=f"imagemagick_lzw_pred{i}",
        series="tiff-lzw",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_im_comp("-compress", "LZW", "-define", f"tiff:predictor={i}"),
        **_BC_IMAGEMAGICK,
    )
    for i in [1, 2, 3]
]
LZW_CONFIGS = LIBTIFF_LZW_CONFIGS + IMAGEMAGICK_LZW_CONFIGS

# --- DEFLATE ---
LIBTIFF_DEFLATE_CONFIGS = [
    _C(
        name=f"libtiff_deflate_pred{i}_s{s}_p{p}",
        series="tiff-deflate",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        subcodec="zlib" if s == 0 else "libdeflate",
        subcodec_index=s,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"zip:{i}:s{s}:p{p}"),
        **_BC_LIBTIFF,
    )
    for i in [1, 2, 3]
    for p in [0, 1, 2, 4, 6, 8, 10]
    for s in [0, 1]  # subcodec, 0=zlib, 1=libdeflate
]

IMAGEMAGICK_DEFLATE_CONFIGS = [
    _C(
        name=f"imagemagick_deflate_pred{i}_p{p}",
        series="tiff-deflate",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        subcodec=None,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "Zip",
            "-define",
            f"tiff:predictor={i}",
            "-quality",
            f"{p * 10}",
        ),
        **_BC_IMAGEMAGICK,
    )
    for i in [1, 2, 3]
    for p in [0, 1, 2, 4, 6, 8, 10]
    # Note: No subcodec available in ImageMagick, so no s=0/1 loop here
]

DEFLATE_CONFIGS = LIBTIFF_DEFLATE_CONFIGS + IMAGEMAGICK_DEFLATE_CONFIGS

# --- LZMA ---
LIBTIFF_LZMA_CONFIGS = [
    _C(
        name=f"libtiff_lzma_pred{i}_p{p}",
        series="tiff-lzma",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"lzma:{i}:p{p}"),
        **_BC_LIBTIFF,
    )
    for i in [1, 2, 3]
    for p in [0, 1, 2, 4, 6, 8, 10]
]
IMAGEMAGICK_LZMA_CONFIGS = [
    _C(
        name=f"imagemagick_lzma_pred{i}_p{p}",
        series="tiff-lzma",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "LZMA",
            "-define",
            f"tiff:predictor={i}",
            "-quality",
            f"{p * 10}",
        ),
        **_BC_IMAGEMAGICK,
    )
    for i in [1, 2, 3]
    for p in [0, 1, 2, 4, 6, 8, 10]
]
LZMA_CONFIGS = LIBTIFF_LZMA_CONFIGS + IMAGEMAGICK_LZMA_CONFIGS

# --- Zstd ---
LIBTIFF_ZSTD_CONFIGS = [
    _C(
        name=f"libtiff_zstd_pred{i}_p{p}",
        series="tiff-zstd",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"zstd:{i}:p{p}"),
        **_BC_LIBTIFF,
    )
    for i in [1, 2, 3]
    for p in [1, 5, 9, 13, 17, 19, 22]
]

IMAGEMAGICK_ZSTD_CONFIGS = [
    _C(
        name=f"imagemagick_zstd_pred{i}_p{p}",
        series="tiff-zstd",
        differencing=_PREDICTOR_NAME_MAP[i],
        predictor=i,
        effort=p,
        allowed_dtypes=DTYPE_FLOAT if i == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "Zstd",
            "-define",
            f"tiff:predictor={i}",
            "-quality",
            f"{p * 5}",
        ),
        **_BC_IMAGEMAGICK,
    )
    for i in [1, 2, 3]
    for p in [1, 5, 9, 13, 17, 19, 22]
]

# --- LERC ---
_LIBTIFF_LERC_SUBCODECS = {
    0: "none",
    1: "deflate",
    2: "zstd",
}

_LERC_COMBINATIONS = []
for p in [1, 5, 8, 10, 12, 14, 16, 18, 20, 22]:
    for s in [0, 1, 2]:
        if s == 1 and p > 12:
            continue  # deflate subcodec only supports p <= 12 for whatever reason, I think a bug in libtiff
        _LERC_COMBINATIONS.append({"z": 0, "s": s, "p": p})

LIBTIFF_LERC_CONFIGS = [
    _C(
        name=f"libtiff_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        subseries="tiff-lerc-libtiff",
        effort=c["p"],
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        subcodec_index=c["s"],
        z_max_error=c["z"],
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", f"lerc:{c['z']}:s{c['s']}:p{c['p']}"),
        **_BC_LIBTIFF,
    )
    for c in _LERC_COMBINATIONS
]


def _resolve_imagemagick_lerc(z, s, p):
    out = [
        "-compress",
        "LERC",
        "-define",
        f"tiff:lerc-maxzerror={z}",
    ]

    if s == 0:
        return tuple(out)
    # Here p must be 1-12
    if s == 1 and p <= 12:
        out += ["-define", f"tiff:lerc-compression={s}"]
        out += ["-define", f"tiff:lerc-zip-quality={p}"]
        return tuple(out)

    if s == 2:  # p 1-22
        out += ["-define", f"tiff:lerc-compression={s}"]
        out += ["-define", f"tiff:lerc-zstd-level={p}"]
        return tuple(out)

    raise ValueError(f"Invalid combination of s={s} and p={p} for LERC in ImageMagick")


IMAGEMAGICK_LERC_CONFIGS = [
    _C(
        name=f"imagemagick_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        effort=c["p"],
        subcodec_index=c["s"],
        z_max_error=c["z"],
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp(
            *_resolve_imagemagick_lerc(c["z"], c["s"], c["p"]),
        ),
        **_BC_IMAGEMAGICK,
    )
    for c in _LERC_COMBINATIONS
]
LERC_CONFIGS = LIBTIFF_LERC_CONFIGS + IMAGEMAGICK_LERC_CONFIGS

# --- SGILOG ---
LIBTIFF_SGILOG_CONFIGS = [
    _C(
        name="libtiff_sgilog",
        series="tiff-sgilog",
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", "sgilog"),
        **_BC_LIBTIFF,
    )
]

# --- RLE ---
RLE_CONFIGS = [
    _C(
        name="libtiff_rle",
        series="tiff-rle",
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", "packbits"),
        **_BC_LIBTIFF,
    ),
    _C(
        name="imagemagick_rle",
        series="tiff-rle",
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp("-compress", "RLE"),
        **_BC_IMAGEMAGICK,
    ),
]

# # TODO: u1/u8 codeccs not set up since we don't really care about those

CONFIGS = (
    []
    + BASELINE_CONFIGS
    + LZW_CONFIGS
    + DEFLATE_CONFIGS
    + LZMA_CONFIGS
    + LERC_CONFIGS
    + LIBTIFF_SGILOG_CONFIGS
    + RLE_CONFIGS
)

In [13]:
# Check duplicate names, happens if we forgot f in front of f-string
names = [c.name for c in CONFIGS]
if len(names) != len(set(names)):
    duplicates = set([name for name in names if names.count(name) > 1])
    raise ValueError(f"Duplicate config names found: {duplicates}")


In [ ]:
BASE_EXTRA = {
    "bulk_runner_kwargs": {"n_jobs": 16, "joblib_kwargs": {"prefer": "processes"}},
}

host = socket.gethostname()
initial_size = sum(f.stat().st_size for f in INPUT_FILES)

tests = []

TEST_ID = f"full_{host}"
TEST_OUTPUT_DIR = (
    OUTPUT_DIR
    / "file-stack"
    / (TEST_ID + "_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    / DATASET
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for config in CONFIGS:
    try:
        if INPUT_DTYPE not in config.allowed_dtypes:
            print("Skipping config", config.name, "for unsupported dtype", INPUT_DTYPE)
            continue

        start_dt = datetime.now(UTC)
        total_time_start = perf_counter()
        extra = BASE_EXTRA.copy()

        # Put in tmpfs to avoid disk I/O bottleneck
        # This is automatically deleted at end of scope/context manager
        with mem_temper.TemporaryDirectory() as compress_temp_dir:
            logging.debug(
                "Created temporary directory for compression: %s", compress_temp_dir
            )
            compress_temp_dir_path = Path(compress_temp_dir)

            # Run compression
            compress_start = perf_counter()
            compressed_files = [
                compress_temp_dir_path / (input_file.name) for input_file in INPUT_FILES
            ]
            config.compressor[0](
                INPUT_FILES,
                compressed_files,
                *config.compressor[1],
                **config.compressor[2],
                **extra["bulk_runner_kwargs"],
            )
            compress_time_seconds = perf_counter() - compress_start

            # Run decompression
            with mem_temper.TemporaryDirectory() as decompress_temp_dir:
                logging.debug(
                    "Created temporary directory for decompression: %s",
                    decompress_temp_dir,
                )
                decompress_temp_dir_path = Path(decompress_temp_dir)
                decompressed_files = [
                    decompress_temp_dir_path / (input_file.name)
                    for input_file in INPUT_FILES
                ]
                decompress_start = perf_counter()
                config.decompressor[0](
                    compressed_files,
                    decompressed_files,
                    *config.decompressor[1],
                    **config.decompressor[2],
                    **extra["bulk_runner_kwargs"],
                )
                decompress_time_seconds = perf_counter() - decompress_start

                # Calculate size before releasing from memory
                uncompressed_size = initial_size
                actual_uncompressed_size = get_dir_size_bytes(decompress_temp_dir_path)
                compressed_size = get_dir_size_bytes(compress_temp_dir_path)

                # Run comparison
                comparisons = config.comparer[0](
                    INPUT_FILES,
                    decompressed_files,
                    *config.comparer[1],
                    **config.comparer[2],
                    **extra["bulk_runner_kwargs"],
                )

                was_actually_lossless = all(comparisons)

        # Get pixel count for bitrate
        n_elements = 0
        for input_file in INPUT_FILES:
            # Note that this does not actually read image data, so not super slow
            with TiffFile(input_file) as tif:
                for page in tif.pages:
                    n_elements += prod(page.shape)

        # Calculate quantities
        compression_rate = uncompressed_size / compress_time_seconds
        decompression_rate = uncompressed_size / decompress_time_seconds

        bitrate_uncompressed = 8 * initial_size / n_elements
        bitrate_compressed = 8 * compressed_size / n_elements

        compression_ratio = uncompressed_size / compressed_size
        space_saving = 1 - 1 / compression_ratio

        end_dt = datetime.now(UTC)
        total_time_seconds = perf_counter() - total_time_start

        test = CompressionTest(
            **config.model_dump(),
            id=config.get_pathlike_str(),
            dataset_name=DATASET,
            dataset_path=DATASET_DIR,
            start_dt=start_dt,
            end_dt=end_dt,
            total_time_seconds=total_time_seconds,
            config=config,
            baseline_test=None,  # TODO: For now
            host=host,
            compress_time_seconds=compress_time_seconds,
            decompress_time_seconds=decompress_time_seconds,
            uncompressed_size=uncompressed_size,
            actual_uncompressed_size=actual_uncompressed_size,
            compressed_size=compressed_size,
            n_elements=n_elements,
            compression_rate=compression_rate,
            decompression_rate=decompression_rate,
            bitrate_uncompressed=bitrate_uncompressed,
            bitrate_compressed=bitrate_compressed,
            compression_ratio=compression_ratio,
            space_saving=space_saving,
            was_actually_lossless=was_actually_lossless,
            runner_extra=extra,
        )

        tests.append(test)

        pprint(test.model_dump())

        output_file = TEST_OUTPUT_DIR / f"test_{test.name}.json"
        with open(output_file, "w") as f:
            f.write(test.model_dump_json(indent=4, fallback=repr))

    except Exception as e:
        print(f"Error running test for config {config.name}: {e}")
        traceback.print_exc()

Error running test for config libtiff_baseline: comptest.types.CompressionTest() got multiple values for keyword argument 'extra'


Traceback (most recent call last):
  File "/tmp/ipykernel_1501993/2473750304.py", line 109, in <module>
    test = CompressionTest(
        **config.model_dump(),
    ...<22 lines>...
        extra=extra,
    )
TypeError: comptest.types.CompressionTest() got multiple values for keyword argument 'extra'


In [ ]:
# Clean up memory
shutil.rmtree(temp_dir_path)